# 08 — Baseline & Regularized Linear Models

**Objective:** Benchmark foundational baseline models:
- DummyRegressor (Mean and Median)
- Linear Regression (OLS)
- Ridge Regression (L2 regularization)
- Lasso Regression (L1 regularization)
- ElasticNet Regression (L1 + L2)


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

def find_root():
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / 'data' / 'raw' / 'gt_2011.csv').exists():
            return parent
    return p

ROOT = find_root()
sys.path.insert(0, str(ROOT))

from src.models.model_factory import create_model
from src.data.loader import FEATURES_A, TARGETS_A
from src.data.splitting import temporal_split
from src.evaluation.metrics import regression_metrics


## 1. Benchmarking Baselines on Development Folds


In [2]:
df = pd.read_parquet(ROOT / "data" / "processed" / "cleaned_master.parquet")
dev, cal, test = temporal_split(df)

baseline_names = ["Dummy_mean", "LinearRegression", "Ridge", "Lasso", "ElasticNet"]
records = []

for name in baseline_names:
    model = create_model(name, quick=True, random_state=42)
    model.fit(dev[FEATURES_A], dev[TARGETS_A].values)
    preds = model.predict(cal[FEATURES_A])
    m_list = regression_metrics(cal[TARGETS_A].values, preds, TARGETS_A)
    for m in m_list:
        records.append({"model": name, "split": "2014_calibration", **m})

df_base = pd.DataFrame(records)
print(df_base[["model", "target", "RMSE", "MAE", "R2"]].to_string(index=False))

out_base = ROOT / "experiments" / "baseline_metrics.csv"
out_base.parent.mkdir(parents=True, exist_ok=True)
df_base.to_csv(out_base, index=False)
print(f"Saved baseline metrics to {out_base.relative_to(ROOT)}")
print("Baseline model benchmarking COMPLETE.")


           model target    RMSE     MAE      R2
      Dummy_mean     CO  2.0112  1.0681 -0.0044
      Dummy_mean    NOX 13.2393 11.4013 -0.7630
LinearRegression     CO  1.5406  0.9190  0.4107
LinearRegression    NOX  8.5505  6.4373  0.2646
           Ridge     CO  1.5313  0.9081  0.4178
           Ridge    NOX  8.6175  6.5353  0.2531
           Lasso     CO  1.4849  0.9061  0.4525
           Lasso    NOX  9.4356  7.6313  0.1045
      ElasticNet     CO  1.4833  0.9024  0.4537
      ElasticNet    NOX 10.4849  8.9822 -0.1057
Saved baseline metrics to experiments\baseline_metrics.csv
Baseline model benchmarking COMPLETE.
